# 1년 생존율은 내리고 5년 생존율은 올랐다? 같은 해 신생기업끼리 다시 읽기

DataPopcorn 테크블로그 글의 재현 노트북입니다. `런타임 → 모두 실행`을 누르면 국가데이터처 「2025년 기업생멸행정통계(잠정)」 보도자료 **<표 19> 연도별 신생기업 생존율**의 행 텍스트에서 시작해, 표를 CSV로 다시 만들고 글에 나온 숫자와 **차트 4개**를 그립니다.

- 데이터: 보도자료 PDF 27쪽(인쇄 21쪽) 표19 영역의 행 텍스트를 그대로 옮긴 `table19_raw_lines.csv`(2026-10-24 조회). API 키가 필요 없습니다.
- 원문 PDF와 기사 본문은 싣지 않았습니다. PDF 주소·SHA-256·쪽 번호는 `source_metadata.json`에 있습니다.
- 단위: 생존율 %, 변화 %p. 관측 연도 2024년 값은 잠정(p)입니다. 발표는 2026년, 통계 제목은 2025년, 생존율의 최신 관측 연도는 2024년입니다.
- 원문 PDF를 직접 내려받아 해시를 확인하고 27쪽을 다시 읽는 코드는 맨 아래 부록에 있고 기본으로 꺼져 있습니다(`RUN_COLLECT = False`).

In [ ]:
# Step 1. 준비: 한글 폰트(Colab이면 나눔고딕 설치)와 데이터 위치
import os, sys, io, re, json, hashlib, urllib.request
import numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt
from matplotlib import font_manager as fm

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    os.system('apt-get -qq install -y fonts-nanum > /dev/null')
for f in ['/usr/share/fonts/truetype/nanum/NanumGothic.ttf', '/System/Library/Fonts/AppleSDGothicNeo.ttc', 'C:/Windows/Fonts/malgun.ttf']:
    if os.path.exists(f):
        fm.fontManager.addfont(f)
        plt.rcParams['font.family'] = fm.FontProperties(fname=f).get_name()
        break
plt.rcParams.update({'axes.unicode_minus': False, 'figure.dpi': 110, 'axes.spines.top': False,
                     'axes.spines.right': False, 'font.size': 11})

# 데이터 위치: 공개 GitHub 저장소 (로컬에서 돌릴 때는 DATA_DIR 환경변수로 폴더 지정)
BASE = os.environ.get('DATA_DIR', 'https://raw.githubusercontent.com/team-datapopcorn/datapopcorn-assets/main/business-survival-cohorts/data')
def read_bytes(rel):
    p = f'{BASE}/{rel}'
    return urllib.request.urlopen(p).read() if p.startswith('http') else open(p, 'rb').read()
def load_csv(rel): return pd.read_csv(io.BytesIO(read_bytes(rel)), encoding='utf-8-sig')
meta = json.loads(read_bytes('source_metadata.json').decode('utf-8'))
print('데이터 경로:', BASE)
print('원문:', meta['dataset']); print('PDF SHA-256:', meta['pdf']['sha256'][:16], '...', '표19 위치: PDF', meta['table19']['pdf_page'], '쪽')

C = {'main': '#E8590C', 'sub': '#1C7ED6', 'gray': '#868E96', 'ink': '#212529'}
SOURCE_NOTE = '자료: 국가데이터처 「2025년 기업생멸행정통계(잠정)」 보도자료 <표 19> (PDF 27쪽, 2026-10-22 발표), 2026-10-24 조회. 2024년 값은 잠정(p).'
def _foot(fig, extra=''):
    fig.text(0.01, -0.02, SOURCE_NOTE + (('\n' + extra) if extra else ''), fontsize=8, color=C['gray'], va='top')

## Step 2. 표19 행 텍스트를 긴 표(CSV)로 바꿉니다

표19는 가로가 생존 기간(1년~7년), 세로가 **관측 연도**(2021~2024)입니다. 여기에 **신생 연도 = 관측 연도 - 생존 기간** 열을 붙여야 각 숫자가 어느 해에 생긴 기업의 비율인지 보입니다. 원문 정의는 `n년 생존율 = (t-n년 신생기업 중 t년까지 생존 기업 수 / t-n년 신생기업 수) x 100`입니다.

표의 첫 블록에는 라벨이 없습니다. 같은 보도자료 표20·21의 '전 산 업' 행과 값이 같아 `전체`로 표기합니다.

In [ ]:
lines = load_csv('table19_raw_lines.csv')
print(lines[['line_no', 'text']].to_string(index=False))

COLUMNS = ['scope', 'observation_year', 'horizon_years', 'birth_year', 'rate_pct', 'pdf_page']
NUM7 = '((?:-?\\d+\\.\\d\\s+){6}-?\\d+\\.\\d)'
ROW_RE = re.compile(r'^(?:(부동산업\s*제외)\s+)?(\d{4})년(p?)\s+' + NUM7 + r'\s*$')
CHG_RE = re.compile(r'^증감\s+' + NUM7 + r'\s*$')
SOURCE_PDF_PAGE = 27
def parse_table19(lines, pdf_page=SOURCE_PDF_PAGE):
    """표19 행 텍스트 → (생존율 긴 표, 공식 증감 행, 잠정 표시 연도).

    잠정 표시 'p'는 위첨자라 추출 도구에 따라 다른 연도 줄에 붙기도 한다(pdftotext -layout은 2023년 줄에 붙음).
    그래서 잠정 연도는 참고로만 돌려주고, 숫자 대조에는 쓰지 않는다.
    '부동산업 제외' 라벨이 처음 나오기 전 행은 전체 산업(원문 표에는 라벨 없음)으로 본다.
    신생 연도 = 관측 연도 - 생존 기간 (원문 정의: n년 생존율 = t-n년 신생기업 중 t년까지 생존 기업 수 / t-n년 신생기업 수)."""
    scope, rates, changes, provisional = '전체', [], [], set()
    for ln in lines:
        s = ' '.join(ln.split())
        m = ROW_RE.match(s)
        if m:
            if m.group(1):
                scope = '부동산업 제외'
            year = int(m.group(2))
            if m.group(3) == 'p':
                provisional.add(year)
            for h, v in enumerate(m.group(4).split(), 1):
                rates.append({'scope': scope, 'observation_year': year, 'horizon_years': h,
                              'birth_year': year - h, 'rate_pct': float(v), 'pdf_page': pdf_page})
            continue
        m = CHG_RE.match(s)
        if m:
            for h, v in enumerate(m.group(1).split(), 1):
                changes.append({'scope': scope, 'horizon_years': h, 'change_pp': float(v), 'pdf_page': pdf_page})
    return pd.DataFrame(rates, columns=COLUMNS), pd.DataFrame(changes), sorted(provisional)

rates, changes, provisional = parse_table19(lines.text.tolist())
print(f'\n파싱 결과 {len(rates)}행 (2개 범위 x 관측 4개 연도 x 7개 기간), 잠정 표시 연도: {provisional}')
print(rates.head(7).to_string(index=False))

In [ ]:
# 검증: 행 수·중복·기대값, 그리고 표의 '증감' 행 = 2024년 값 - 2023년 값인지
EXPECTED = {('전체', 2024, 1): 63.9, ('전체', 2024, 5): 36.5, ('전체', 2023, 1): 64.4, ('전체', 2023, 5): 36.4, ('전체', 2021, 2): 54.2, ('전체', 2022, 3): 46.9, ('전체', 2023, 4): 41.0, ('부동산업 제외', 2024, 1): 63.7, ('부동산업 제외', 2024, 5): 36.0, ('부동산업 제외', 2023, 1): 64.0, ('부동산업 제외', 2023, 5): 35.3}
def validate(rates, changes):
    """행 수, 키 중복, 기대값, 공식 증감 행과 계산 차이를 대조한다. 실패하면 AssertionError."""
    assert len(rates) == 56, len(rates)                       # 2 범위 x 4 관측연도 x 7 기간
    assert not rates.duplicated(['scope', 'observation_year', 'horizon_years']).any()
    assert set(rates.scope) == {'전체', '부동산업 제외'}
    assert sorted(rates.observation_year.unique()) == [2021, 2022, 2023, 2024]
    assert (rates.birth_year == rates.observation_year - rates.horizon_years).all()
    key = rates.set_index(['scope', 'observation_year', 'horizon_years']).rate_pct
    for k, v in EXPECTED.items():
        assert key[k] == v, (k, key[k], v)
    assert len(changes) == 14
    w = rates.pivot_table(index=['scope', 'horizon_years'], columns='observation_year', values='rate_pct')
    diff = (w[2024] - w[2023]).round(1)
    off = changes.set_index(['scope', 'horizon_years']).change_pp
    mism = (diff - off.reindex(diff.index)).abs().round(1)
    assert (mism == 0).all(), mism[mism != 0]
    return diff

calc_diff = validate(rates, changes)
print('표의 증감 행과 계산 차이 14칸 모두 일치')
published = load_csv('survival_rates.csv')
print('공개 CSV(survival_rates.csv)와 동일:', published[COLUMNS].equals(rates[COLUMNS]))
assert published[COLUMNS].equals(rates[COLUMNS])
k = rates.set_index(['scope', 'observation_year', 'horizon_years']).rate_pct

## Step 3. 최신 1년·5년 생존율 변화에 신생 연도를 붙입니다 (차트 1)

뉴스의 "1년 생존율 63.9%, 0.5%p 하락"과 "5년 생존율 36.5%, 0.1%p 상승"은 관측 연도 2023년과 2024년을 비교한 값입니다. 신생 연도를 붙이면 네 숫자의 분모 집단이 모두 다릅니다.

In [ ]:
for h in (1, 5):
    a, b = k[('전체', 2023, h)], k[('전체', 2024, h)]
    print(f'{h}년 생존율: 2023년 관측 {a:.1f}% ({2023 - h}년 신생) -> 2024년 관측 {b:.1f}% ({2024 - h}년 신생), 변화 {b - a:+.1f}%p')

def chart_latest_changes(rates, scope='전체'):
    import matplotlib.pyplot as plt
    k = rates[rates.scope == scope].set_index(['observation_year', 'horizon_years']).rate_pct
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.8), sharey=True)
    d1, d5 = k[(2024, 1)] - k[(2023, 1)], k[(2024, 5)] - k[(2023, 5)]
    for ax, h, col in [(axes[0], 1, C['main']), (axes[1], 5, C['sub'])]:
        a, b = k[(2023, h)], k[(2024, h)]
        labels = [f'2023년 관측\n({2023 - h}년 신생)', f'2024년p 관측\n({2024 - h}년 신생)']
        ax.bar(labels, [a, b], color=[C['gray'], col], width=0.55)
        for i, v in enumerate([a, b]):
            ax.text(i, v + 1.2, f'{v:.1f}%', ha='center', fontsize=13, fontweight='bold')
        ax.set_title(f'{h}년 생존율  {b - a:+.1f}%p', loc='left', fontsize=13, color=col)
        ax.set_ylim(0, 80)
    axes[0].set_ylabel('생존율 (%)')
    fig.suptitle(f'1년 생존율은 {abs(d1):.1f}%p 내리고 5년은 {d5:.1f}%p 올랐지만, 네 막대는 모두 다른 해에 생긴 기업이다\n'
                 '전체 산업 · 관측 2023년 → 2024년(p) · 괄호 = 그 비율의 분모가 된 신생 연도',
                 x=0.01, ha='left', fontsize=12)
    plt.tight_layout(rect=(0, 0, 1, 0.93))
    _foot(fig)
    return fig

fig = chart_latest_changes(rates)
plt.show()

## Step 4. 2024년 한 해의 1~7년 생존율은 '한 집단의 곡선'이 아닙니다 (차트 2)

관측 연도 2024년 행의 7개 값은 2023년부터 2017년까지 서로 다른 해에 생긴 7개 집단의 비율입니다.

In [ ]:
cs = rates[(rates.scope == '전체') & (rates.observation_year == 2024)]
print(cs[['horizon_years', 'birth_year', 'rate_pct']].to_string(index=False))

def chart_cross_section(rates, scope='전체', year=2024):
    import matplotlib.pyplot as plt
    r = rates[(rates.scope == scope) & (rates.observation_year == year)].sort_values('horizon_years')
    fig, ax = plt.subplots(figsize=(11, 4.9))
    colors = [C['main'] if h == 1 else C['sub'] if h == 5 else C['gray'] for h in r.horizon_years]
    labels = [f'{h}년\n({b}년 신생)' for h, b in zip(r.horizon_years, r.birth_year)]
    ax.bar(labels, r.rate_pct, color=colors, width=0.62)
    for i, v in enumerate(r.rate_pct):
        ax.text(i, v + 1.0, f'{v:.1f}', ha='center', fontsize=11)
    ax.set_ylim(0, 75); ax.set_ylabel('생존율 (%)')
    ax.text(6.4, 68, '막대 7개 = 신생 연도가 다른 7개 집단\n한 집단을 7년 따라간 곡선이 아닙니다', ha='right',
            fontsize=10, color=C['ink'])
    ax.set_title(f'{year}년에 한꺼번에 잰 1~7년 생존율: {r.rate_pct.iloc[0]:.1f}%에서 {r.rate_pct.iloc[-1]:.1f}%까지, 막대마다 신생 연도가 다르다\n'
                 f'전체 산업 · 관측 {year}년(p) 한 해의 단면', loc='left', fontsize=12)
    plt.tight_layout(); _foot(fig)
    return fig

fig = chart_cross_section(rates)
plt.show()

## Step 5. 같은 2019년 신생기업만 따라갑니다 (차트 3)

신생 연도가 2019인 칸만 고르면 (2021년 관측, 2년), (2022년, 3년), (2023년, 4년), (2024년, 5년)입니다. 표19의 관측 범위는 2021~2024년이라 1년 생존율(2020년 관측)은 이번 분석에 넣지 않았습니다. 네 값은 모두 이번 발표본 표19의 값이며, 원문 유의사항(PDF 83쪽)대로 과거 수치는 산업분류 갱신 등으로 바뀔 수 있습니다.

여기서 36.5 / 63.9 처럼 같은 해 행의 두 값을 나누면 안 됩니다. 분자와 분모가 다른 해에 생긴 집단이라 '1년 살아남은 기업이 5년까지 갈 확률'(조건부 생존확률)이 아닙니다.

In [ ]:
def cohort_path(rates, birth_year=2019, scope='전체', horizons=(2, 3, 4, 5)):
    p = rates[(rates.scope == scope) & (rates.birth_year == birth_year) & rates.horizon_years.isin(horizons)]
    return p.sort_values('observation_year').reset_index(drop=True)

p19 = cohort_path(rates)
print(p19[['observation_year', 'horizon_years', 'birth_year', 'rate_pct']].to_string(index=False))
wrong = k[('전체', 2024, 5)] / k[('전체', 2024, 1)] * 100
print(f'잘못된 계산 예시: 36.5 / 63.9 = {wrong:.1f}%  <- 2019년 신생(분자)과 2023년 신생(분모)을 섞은 값, 조건부 생존확률 아님')

def chart_2019_cohort(rates):
    import matplotlib.pyplot as plt
    p = cohort_path(rates)
    fig, ax = plt.subplots(figsize=(11, 4.9))
    x = [f'{o}년 관측\n{h}년 생존율' for o, h in zip(p.observation_year, p.horizon_years)]
    ax.plot(x, p.rate_pct, color=C['main'], lw=2.8, marker='o', ms=9)
    for i, v in enumerate(p.rate_pct):
        ax.text(i, v + 1.6, f'{v:.1f}%', ha='center', fontsize=13, fontweight='bold')
    ax.set_ylim(0, 65); ax.set_ylabel('2019년 신생기업 중 생존 비율 (%)')
    path = ' → '.join(f'{h}년 {v:.1f}%' for h, v in zip(p.horizon_years, p.rate_pct))
    ax.set_title(f'같은 2019년 신생기업을 따라가면 {path}\n'
                 '전체 산업 · 관측 2021~2024년(p)의 각 행에서 신생 연도가 2019인 칸만 선택', loc='left', fontsize=12)
    plt.tight_layout()
    _foot(fig, '표19의 관측 범위는 2021~2024년이라 2019년 신생의 1년 생존율(2020년 관측)은 이 차트에 넣지 않았습니다. '
          '모든 값은 이번 발표본 기준이며 과거 수치는 갱신될 수 있습니다.')
    return fig

fig = chart_2019_cohort(rates)
plt.show()

## Step 6. 부동산업을 빼고 다시 봅니다 (차트 4)

보도자료는 전체 산업과 함께 '부동산업 제외' 생존율도 냅니다. 같은 비교를 두 범위에서 해 보고, 방향과 폭이 어떻게 달라지는지 봅니다. 폭이 왜 다른지(업종 구성 등)는 이 표만으로 판단할 수 없습니다.

In [ ]:
for sc in ['전체', '부동산업 제외']:
    print(f"{sc}: 1년 {calc_diff[(sc, 1)]:+.1f}%p, 5년 {calc_diff[(sc, 5)]:+.1f}%p")

def chart_scope_sensitivity(rates):
    import matplotlib.pyplot as plt
    import numpy as np
    w = rates.pivot_table(index=['scope', 'horizon_years'], columns='observation_year', values='rate_pct')
    d = (w[2024] - w[2023]).round(1)
    groups = [(1, '1년 생존율 변화\n(2022년 신생 → 2023년 신생)'), (5, '5년 생존율 변화\n(2018년 신생 → 2019년 신생)')]
    fig, ax = plt.subplots(figsize=(11, 4.9))
    xs = np.arange(2); bw = 0.34
    for j, (sc, col) in enumerate([('전체', C['main']), ('부동산업 제외', C['sub'])]):
        vals = [d[(sc, h)] for h, _ in groups]
        ax.bar(xs + (j - 0.5) * bw, vals, bw, color=col, label=sc)
        for x, v in zip(xs + (j - 0.5) * bw, vals):
            ax.text(x, v + (0.05 if v >= 0 else -0.05), f'{v:+.1f}%p', ha='center',
                    va='bottom' if v >= 0 else 'top', fontsize=12, fontweight='bold')
    ax.axhline(0, color=C['ink'], lw=0.8)
    ax.set_xticks(xs); ax.set_xticklabels([g for _, g in groups])
    ax.set_ylim(-0.9, 1.0); ax.set_ylabel('관측 2023 → 2024년(p) 변화 (%p)')
    ax.legend(frameon=False, loc='upper left')
    ax.set_title('부동산업을 빼도 방향은 같다: 1년은 하락, 5년은 상승. 폭은 범위에 따라 달라진다\n'
                 '전체 산업 vs 부동산업 제외 · 관측 2023년 → 2024년(p)', loc='left', fontsize=12)
    plt.tight_layout()
    _foot(fig, '범위에 따른 폭 차이의 원인(업종 구성 등)은 이 표만으로 판단할 수 없습니다.')
    return fig

fig = chart_scope_sensitivity(rates)
plt.show()

## 해석할 때 주의할 점

- 생존율은 '폐업 신고를 안 했다'가 아니라, 신생 이후 관측 연도까지 소멸되지 않고 지속적으로 존속했다는 뜻입니다. 이익을 냈다는 뜻도 아닙니다.
- 신생기업은 매출액 또는 상용근로자가 새로 생긴 영리기업이며, 기존 기업이라도 대표자·주소·산업 중분류 중 2개 이상이 동시에 바뀌면 신생으로 분류됩니다(PDF 81쪽).
- 63.9%는 비율입니다. '100곳 중 63.9곳'은 비율을 풀어 쓴 설명이지 실제로 센 100곳이 아닙니다. 이 표에는 기업 수가 없습니다.
- 2024년 값은 잠정이며, 2026년 9월까지 입수된 행정자료 기준입니다. 확정 때 바뀔 수 있습니다.
- 원문 유의사항(PDF 83쪽): 사업자등록자료 산업분류 갱신 등으로 과거 일부 수치가 바뀔 수 있고, 2023년부터 법인세 DB 구축 방법 개선으로 그전까지 일부 빠졌던 금융·보험업 기업 자료가 추가됐습니다. 연도 간 비교는 이 점을 감안해야 합니다.
- 변화의 원인(경기·금리·정책 등)은 이 표로 판단하지 않습니다.

## 부록: 원문 PDF를 직접 받아 해시를 확인하고 27쪽을 다시 읽기

기본으로 꺼져 있습니다(`RUN_COLLECT = False`). 키가 필요 없지만, 사이트 구조나 접속 환경(해외 IP 등)에 따라 다운로드가 막힐 수 있습니다. PDF 텍스트 추출 도구마다 위첨자 'p'가 붙는 줄이나 띄어쓰기('부동산업제외')가 달라질 수 있어, 숫자만 대조합니다.

In [ ]:
RUN_COLLECT = False
PDF_URL = meta['pdf']['url']
def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(1 << 20), b''):
            h.update(b)
    return h.hexdigest()
def table19_block(page_text):
    """PDF 27쪽 텍스트에서 '표 19>' 줄부터 '그림 1>' 직전 줄까지를 잘라낸다.
    추출 도구에 따라 '<'가 다른 줄로 떨어지거나 '부동산업제외'처럼 띄어쓰기가 사라지므로 느슨하게 찾는다."""
    lines = page_text.splitlines()
    start = next(i for i, l in enumerate(lines) if '표 19>' in l)
    end = next(i for i, l in enumerate(lines) if i > start and '그림 1>' in l)
    return [l.rstrip() for l in lines[start:end]]

def collect_and_check(url=PDF_URL, out='business-demography-2025.pdf'):
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with open(out, 'wb') as f:
        f.write(urllib.request.urlopen(req, timeout=60).read())
    digest = sha256(out)
    print('다운로드', os.path.getsize(out), 'bytes, SHA-256 일치:', digest == meta['pdf']['sha256'])
    try:
        from pypdf import PdfReader
    except ImportError:
        os.system(f'{sys.executable} -m pip install -q pypdf'); from pypdf import PdfReader
    reader = PdfReader(out)
    print('쪽 수:', len(reader.pages))
    text = reader.pages[meta['table19']['pdf_page'] - 1].extract_text()
    r2, c2, prov2 = parse_table19(table19_block(text))
    same = r2[COLUMNS].equals(rates[COLUMNS]) and c2.equals(changes)
    print('27쪽 재파싱 결과가 CSV와 숫자까지 동일:', same, '/ 잠정 표시가 붙은 줄:', prov2)
    return digest == meta['pdf']['sha256'] and same

if RUN_COLLECT:
    assert collect_and_check()